In [1]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "983f910a",
   "metadata": {},
   "source": [
    "# Stuff+ / Location+ / Pitching+ con Bayesian Resolvability — Diablos Rojos (ISAC 2026)\n",
    "\n",
    "**Entregable 1:** Stuff+ por pitcheo, pitcher y arsenal, y comparación CDMX (altitud extrema) vs. otros estadios.\n",
    "\n",
    "**Innovación:** Integración del modelo Bayesball de resolvabilidad perceptual del bateador en altitud.\n",
    "\n",
    "| Paso | Qué hace |\n",
    "|---|---|\n",
    "| 0 | Configuración y carga de datos con Polars |\n",
    "| 1 | Valor en carreras de cada pitcheo (según el conteo) |\n",
    "| 2 | Variables: físicas, de arsenal, de contexto y bayesianas |\n",
    "| 3 | Variable objetivo descompuesta: 5 resultados posibles + calidad de bola en juego |\n",
    "| 4 | Entrenamiento XGBoost con validación *pitcher holdout* |\n",
    "| 5 | Carreras esperadas (xRV) → escala 100 + 10·z |\n",
    "| 6 | Validación: calidad, estabilidad, capacidad predictiva |\n",
    "| 7 | Efecto altitud: contrafactual físicamente consistente |\n",
    "| 8 | Bayesian Resolvability: modelo perceptual del bateador |\n",
    "| 9 | Agregación por pitcher y por arsenal, y exportación |\n",
    "\n",
    "**Kernel:** Python 3.12, Polars para datos, XGBoost para modelos.\n",
    "\n",
    "**Tiempo:** la primera vez el Paso 4 entrena 30+ modelos (~30–40 min). Los resultados se guardan en `modelos/`; las siguientes veces se cargan en segundos."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f97a20ca",
   "metadata": {},
   "source": [
    "## Paso 0: Configuración con Polars"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "0133c718",
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import time\n",
    "import numpy as np\n",
    "import polars as pl\n",
    "import xgboost as xgb\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.model_selection import GroupKFold\n",
    "from sklearn.metrics import log_loss, mean_squared_error\n",
    "\n",
    "sns.set_theme(style='whitegrid')\n",
    "\n",
    "# Rutas del proyecto\n",
    "CARPETA = \"/Users/ariegoldzweig/Documents/GitHub/2026-ISAC-Hackathon---Diablos-Rojos-WAR_ROOM\"\n",
    "CARPETA_MODELOS = os.path.join(CARPETA, \"modelos\")\n",
    "CARPETA_RESULTADOS = os.path.join(CARPETA, \"resultados\")\n",
    "os.makedirs(CARPETA_MODELOS, exist_ok=True)\n",
    "os.makedirs(CARPETA_RESULTADOS, exist_ok=True)\n",
    "\n",
    "REENTRENAR = False\n",
    "SEMILLA = 42\n",
    "\n",
    "# Cargar datos con Polars\n",
    "df = pl.read_pickle(os.path.join(CARPETA, \"data/raw/stuff_model_df.pkl\"))\n",
    "print(f\"Shape: {df.shape} | xgboost {xgb.__version__} | polars {pl.__version__}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "15c70518",
   "metadata": {},
   "source": [
    "## Paso 1: Valor en carreras de cada pitcheo"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "e6d4f1d8",
   "metadata": {},
   "outputs": [],
   "source": [
    "# ¿Este pitcheo TERMINA la aparición al bat?\n",
    "termina_pa = (\n",
    "    df['KorBB'].is_in(['Strikeout', 'Walk'])\n",
    "    | df['PitchCall'].is_in(['InPlay', 'HitByPitch'])\n",
    ")\n",
    "\n",
    "# ¿Este pitcheo EMPIEZA una aparición nueva?\n",
    "empieza_pa = (\n",
    "    (df['game_anon_id'] != df['game_anon_id'].shift())\n",
    "    | (df['Inning'] != df['Inning'].shift())\n",
    "    | (df['Top/Bottom'] != df['Top/Bottom'].shift())\n",
    "    | (df['batter_anon_id'] != df['batter_anon_id'].shift())\n",
    "    | termina_pa.shift(fill_value=False)\n",
    ")\n",
    "\n",
    "df = df.with_columns(pa_id=empieza_pa.cum_sum())\n",
    "print(f\"Apariciones al bat: {df['pa_id'].n_unique():,}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "id": "95085b5b",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Valor en carreras de cada evento\n",
    "VALOR_EVENTO = {'Walk': 0.31, 'HitByPitch': 0.33, 'Single': 0.46, 'Error': 0.46,\n",
    "                'Double': 0.77, 'Triple': 1.05, 'HomeRun': 1.40,\n",
    "                'Out': -0.27, 'Strikeout': -0.27, 'FieldersChoice': -0.27, 'Sacrifice': -0.27}\n",
    "\n",
    "df = df.with_columns(\n",
    "    evento=pl.when(df['KorBB'] == 'Strikeout').then(pl.lit('Strikeout'))\n",
    "             .when(df['KorBB'] == 'Walk').then(pl.lit('Walk'))\n",
    "             .when(df['PitchCall'] == 'HitByPitch').then(pl.lit('HitByPitch'))\n",
    "             .when(df['PitchCall'] == 'InPlay').then(df['play_result'])\n",
    "             .otherwise(None)\n",
    ")\n",
    "df = df.with_columns(evento=pl.when(termina_pa).then(df['evento']).otherwise(None))\n",
    "df = df.with_columns(rv_evento=df['evento'].replace_strict(VALOR_EVENTO, default=None))\n",
    "\n",
    "print(df['evento'].value_counts().sort('count', descending=True))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "6cd8d3d5",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Valor de cada conteo\n",
    "valor_final_pa = df.filter(df['rv_evento'].is_not_null()).group_by('pa_id').agg(pl.col('rv_evento').last())\n",
    "df = df.join(valor_final_pa, on='pa_id', how='left', suffix='_pa')\n",
    "df = df.rename({'rv_evento_pa': 'rv_pa'})\n",
    "\n",
    "conteo_valido = df['rv_pa'].is_not_null() & (df['Balls'] <= 3) & (df['Strikes'] <= 2)\n",
    "valor_conteo = (\n",
    "    df.filter(conteo_valido)\n",
    "    .unique(['pa_id', 'Balls', 'Strikes'])\n",
    "    .group_by(['Balls', 'Strikes'])\n",
    "    .agg(pl.col('rv_pa').mean())\n",
    ")\n",
    "\n",
    "df = df.join(valor_conteo, on=['Balls', 'Strikes'], how='left', suffix='_conteo')\n",
    "df = df.rename({'rv_pa_conteo': 'v_conteo'})\n",
    "\n",
    "print(\"Tabla de valor por conteo:\")\n",
    "print(valor_conteo.pivot(on='Strikes', index='Balls', values='rv_pa').sort('Balls'))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 5,
   "id": "4336cf31",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Valor del conteo siguiente\n",
    "df = df.with_columns(\n",
    "    v_conteo_sig=pl.col('v_conteo').shift(-1).over('pa_id')\n",
    ")\n",
    "\n",
    "df = df.with_columns(\n",
    "    rv_pitcheo=pl.when(termina_pa)\n",
    "                 .then(df['rv_evento'] - df['v_conteo'])\n",
    "                 .otherwise(df['v_conteo_sig'] - df['v_conteo'])\n",
    ")\n",
    "\n",
    "print(f\"Promedio general: {df['rv_pitcheo'].mean():.3f}\")\n",
    "print(df.group_by('PitchCall').agg([pl.col('rv_pitcheo').mean(), pl.col('rv_pitcheo').count()]).sort('count', descending=True))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "4e1631af",
   "metadata": {},
   "source": [
    "## Paso 2: Variables"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 6,
   "id": "0fa71998",
   "metadata": {},
   "outputs": [],
   "source": [
    "TIPOS_MODELO = ['Four-Seam', 'Sinker', 'Cutter', 'Slider', 'Curveball', 'Changeup', 'Splitter']\n",
    "\n",
    "df = df.with_columns(tipo=pl.col('AutoPitchType').cast(pl.Utf8).replace({'Sweeper': 'Slider'}))\n",
    "df = df.with_columns(pitcheo_anterior=pl.col('tipo').shift().over('pa_id').fill_null('Primero'))\n",
    "\n",
    "valido = (\n",
    "    df['tipo'].is_in(TIPOS_MODELO)\n",
    "    & df['PitcherThrows'].is_in(['Left', 'Right'])\n",
    "    & df['BatterSide'].is_in(['Left', 'Right'])\n",
    "    & (df['PitchCall'] != 'Undefined')\n",
    "    & df['rv_pitcheo'].is_not_null()\n",
    "    & df['pitcher_anon_id'].is_not_null()\n",
    ")\n",
    "\n",
    "d = df.filter(valido).with_row_index('idx')\n",
    "print(f\"Pitcheos en el modelo: {len(d):,} de {len(df):,} ({len(d) / len(df):.1%})\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 7,
   "id": "467720ef",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Variables físicas reflejadas para zurdos\n",
    "zurdo = d['PitcherThrows'] == 'Left'\n",
    "\n",
    "d = d.with_columns([\n",
    "    hb=pl.when(zurdo).then(-d['HorzBreak']).otherwise(d['HorzBreak']),\n",
    "    rel_side=pl.when(zurdo).then(-d['RelSide']).otherwise(d['RelSide']),\n",
    "])\n",
    "\n",
    "eje = pl.when(zurdo).then(360 - d['SpinAxis']).otherwise(d['SpinAxis'])\n",
    "d = d.with_columns([\n",
    "    eje_sin=(eje * np.pi / 180).sin(),\n",
    "    eje_cos=(eje * np.pi / 180).cos(),\n",
    "])\n",
    "\n",
    "print(d.select(['hb', 'rel_side', 'eje_sin', 'eje_cos']).describe())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "id": "95100ab9",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Variables de arsenal (relativas a la recta primaria)\n",
    "conteo_tipos = d.group_by(['pitcher_anon_id', 'year', 'tipo']).agg(pl.col('PitchUID').count().alias('n'))\n",
    "\n",
    "def elegir_recta(grupo):\n",
    "    for candidatos in (['Four-Seam', 'Sinker'], ['Cutter']):\n",
    "        x = grupo.filter(pl.col('tipo').is_in(candidatos))\n",
    "        if len(x) > 0:\n",
    "            return x.sort('n')['tipo'][-1]\n",
    "    return None\n",
    "\n",
    "recta_primaria = conteo_tipos.group_by(['pitcher_anon_id', 'year']).agg(\n",
    "    pl.col('tipo').apply(elegir_recta).alias('fb_tipo')\n",
    ")\n",
    "d = d.join(recta_primaria, on=['pitcher_anon_id', 'year'], how='left')\n",
    "\n",
    "# Promedios de la recta por juego y temporada\n",
    "COLS_RECTA = {'RelSpeed': 'fb_velo', 'InducedVertBreak': 'fb_ivb', 'hb': 'fb_hb',\n",
    "              'RelHeight': 'fb_rel_h', 'rel_side': 'fb_rel_s'}\n",
    "es_recta = d['tipo'] == d['fb_tipo']\n",
    "\n",
    "fb_juego = d.filter(es_recta).group_by(['pitcher_anon_id', 'game_anon_id']).agg(\n",
    "    [pl.col(c).mean().alias(n) for c, n in COLS_RECTA.items()]\n",
    ")\n",
    "fb_temp = d.filter(es_recta).group_by(['pitcher_anon_id', 'year']).agg(\n",
    "    [pl.col(c).mean().alias(n) for c, n in COLS_RECTA.items()]\n",
    ")\n",
    "\n",
    "d = d.join(fb_juego, on=['pitcher_anon_id', 'game_anon_id'], how='left')\n",
    "d = d.join(fb_temp, on=['pitcher_anon_id', 'year'], how='left', suffix='_temp')\n",
    "\n",
    "for c in COLS_RECTA.values():\n",
    "    d = d.with_columns(pl.col(c).fill_null(pl.col(f'{c}_temp')))\n",
    "\n",
    "# Diferencias contra la recta\n",
    "d = d.with_columns([\n",
    "    velo_diff_vs_fb=pl.col('RelSpeed') - pl.col('fb_velo'),\n",
    "    ivb_diff_vs_fb=pl.col('InducedVertBreak') - pl.col('fb_ivb'),\n",
    "    hb_diff_vs_fb=pl.col('hb') - pl.col('fb_hb'),\n",
    "    dist_salida_fb=((pl.col('RelHeight') - pl.col('fb_rel_h'))**2 + (pl.col('rel_side') - pl.col('fb_rel_s'))**2).sqrt(),\n",
    "])\n",
    "\n",
    "uso = d.group_by(['pitcher_anon_id', 'year', 'tipo']).agg(pl.col('PitchUID').count().alias('uso_n'))\n",
    "uso_total = d.group_by(['pitcher_anon_id', 'year']).agg(pl.col('PitchUID').count().alias('total'))\n",
    "uso = uso.join(uso_total, on=['pitcher_anon_id', 'year'])\n",
    "uso = uso.with_columns(uso_pct=pl.col('uso_n') / pl.col('total'))\n",
    "d = d.join(uso.select(['pitcher_anon_id', 'year', 'tipo', 'uso_pct']), on=['pitcher_anon_id', 'year', 'tipo'], how='left')\n",
    "\n",
    "print(f\"Pitcheos sin referencia de recta: {d['fb_velo'].null_count() / len(d):.2%}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 9,
   "id": "57deb48b",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Variables de contexto\n",
    "d = d.with_columns([\n",
    "    altitud=pl.col('altitude_category').cast(pl.Utf8).replace({'No Altitude': 0, 'Medium Altitude': 1, 'Extreme Altitude': 2}),\n",
    "    mano_derecha=(pl.col('PitcherThrows') == 'Right').cast(pl.Int32),\n",
    "    mismo_lado=(pl.col('PitcherThrows') == pl.col('BatterSide')).cast(pl.Int32),\n",
    "    loc_x=pl.when(pl.col('BatterSide') == 'Right').then(-pl.col('PlateLocSide')).otherwise(pl.col('PlateLocSide')),\n",
    "])\n",
    "\n",
    "print(f\"loc_x promedio de pelotazos: {d.filter(pl.col('PitchCall') == 'HitByPitch')['loc_x'].mean():.2f}\")\n",
    "print(d['altitud'].value_counts().sort('altitud'))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "id": "3e9556f0",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Variables por capa\n",
    "FISICAS = ['RelSpeed', 'SpinRate', 'eje_sin', 'eje_cos', 'InducedVertBreak', 'hb',\n",
    "           'Extension', 'RelHeight', 'rel_side']\n",
    "ARSENAL = ['velo_diff_vs_fb', 'ivb_diff_vs_fb', 'hb_diff_vs_fb', 'dist_salida_fb', 'uso_pct']\n",
    "UBICACION = ['loc_x', 'PlateLocHeight']\n",
    "CONTEO = ['Balls', 'Strikes']\n",
    "\n",
    "VARIABLES = {\n",
    "    'stuff': FISICAS + ARSENAL + ['mano_derecha', 'altitud'],\n",
    "    'location': UBICACION + CONTEO + ['tipo', 'mismo_lado', 'mano_derecha', 'altitud'],\n",
    "    'pitching': FISICAS + ARSENAL + UBICACION + CONTEO + ['tipo', 'mismo_lado', 'mano_derecha', 'altitud',\n",
    "                                                          'pitcheo_anterior', 'Outs', 'Inning'],\n",
    "}\n",
    "\n",
    "for capa, cols in VARIABLES.items():\n",
    "    print(f\"{capa:9s} {len(cols):2d} variables\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e5ab1bb0",
   "metadata": {},
   "source": [
    "## Paso 3: Variable objetivo descompuesta"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 11,
   "id": "7d073819",
   "metadata": {},
   "outputs": [],
   "source": [
    "CLASES = ['bola', 'strike_cantado', 'whiff', 'foul', 'en_juego']\n",
    "MAPA_RESULTADO = {'BallCalled': 'bola', 'BallinDirt': 'bola', 'HitByPitch': 'bola',\n",
    "                  'StrikeCalled': 'strike_cantado', 'StrikeSwinging': 'whiff',\n",
    "                  'FoulBall': 'foul', 'FoulBallNotFieldable': 'foul', 'FoulBallFieldable': 'foul',\n",
    "                  'InPlay': 'en_juego'}\n",
    "\n",
    "d = d.with_columns(resultado=pl.col('PitchCall').replace_strict(MAPA_RESULTADO, default=None))\n",
    "d = d.with_columns(y=pl.col('resultado').replace_strict({c: i for i, c in enumerate(CLASES)}))\n",
    "en_juego = (d['resultado'] == 'en_juego') & d['rv_evento'].is_not_null()\n",
    "\n",
    "# Valor en carreras de cada resultado por conteo\n",
    "rv_clase_conteo = d.group_by(['resultado', 'Balls', 'Strikes']).agg(pl.col('rv_pitcheo').mean())\n",
    "rv_clase_liga = d.group_by('resultado').agg(pl.col('rv_pitcheo').mean())\n",
    "v_conteo_bip_liga = d.filter(en_juego)['v_conteo'].mean()\n",
    "\n",
    "print(d['resultado'].value_counts(normalize=True).sort('resultado'))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "466cefe5",
   "metadata": {},
   "source": [
    "## Paso 4: Entrenamiento con pitcher holdout"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 12,
   "id": "09cf67f4",
   "metadata": {},
   "outputs": [],
   "source": [
    "PARAMS_BASE = dict(\n",
    "    n_estimators=1000,\n",
    "    learning_rate=0.1,\n",
    "    max_depth=6,\n",
    "    min_child_weight=50,\n",
    "    subsample=0.8,\n",
    "    colsample_bytree=0.8,\n",
    "    tree_method='hist',\n",
    "    max_bin=128,\n",
    "    enable_categorical=True,\n",
    "    early_stopping_rounds=30,\n",
    "    n_jobs=4,\n",
    "    random_state=SEMILLA,\n",
    ")\n",
    "\n",
    "grupos = d['pitcher_anon_id'].to_numpy()\n",
    "bloques = list(GroupKFold(n_splits=5).split(np.zeros(len(d)), groups=grupos))\n",
    "\n",
    "def separar_early_stopping(idx_entrena, fraccion=0.1, semilla=SEMILLA):\n",
    "    rng = np.random.default_rng(semilla)\n",
    "    pitchers = np.unique(grupos[idx_entrena])\n",
    "    apartados = rng.choice(pitchers, size=int(len(pitchers) * fraccion), replace=False)\n",
    "    es_val = np.isin(grupos[idx_entrena], apartados)\n",
    "    return idx_entrena[~es_val], idx_entrena[es_val]\n",
    "\n",
    "def entrenar_modelo(X, y, idx_entrena, tipo):\n",
    "    idx_fit, idx_val = separar_early_stopping(idx_entrena)\n",
    "    if tipo == 'resultado':\n",
    "        modelo = xgb.XGBClassifier(objective='multi:softprob', eval_metric='mlogloss', **PARAMS_BASE)\n",
    "    else:\n",
    "        modelo = xgb.XGBRegressor(objective='reg:squarederror', eval_metric='rmse', **PARAMS_BASE)\n",
    "    modelo.fit(X[idx_fit], y[idx_fit], eval_set=[(X[idx_val], y[idx_val])], verbose=False)\n",
    "    return modelo\n",
    "\n",
    "def entrenar_capa(capa):\n",
    "    X = d.select(VARIABLES[capa]).to_numpy()\n",
    "    y_res = d['y'].to_numpy()\n",
    "    y_bip = d['rv_evento'].to_numpy()\n",
    "    idx_bip = np.flatnonzero(en_juego.to_numpy())\n",
    "    prob_oof = np.zeros((len(d), len(CLASES)), dtype=np.float32)\n",
    "    bip_oof = np.zeros(len(d), dtype=np.float32)\n",
    "    modelos = []\n",
    "    for i, (entrena, prueba) in enumerate(bloques):\n",
    "        t0 = time.time()\n",
    "        m_res = entrenar_modelo(X, y_res, entrena, 'resultado')\n",
    "        m_bip = entrenar_modelo(X, y_bip, np.intersect1d(entrena, idx_bip), 'bip')\n",
    "        prob_oof[prueba] = m_res.predict_proba(X[prueba])\n",
    "        bip_oof[prueba] = m_bip.predict(X[prueba])\n",
    "        m_res.save_model(os.path.join(CARPETA_MODELOS, f\"{capa}_resultado_{i}.json\"))\n",
    "        m_bip.save_model(os.path.join(CARPETA_MODELOS, f\"{capa}_bip_{i}.json\"))\n",
    "        modelos.append((m_res, m_bip))\n",
    "        print(f\"  {capa} bloque {i + 1}/5: {m_res.best_iteration + 1} árboles resultado, \"\n",
    "              f\"{m_bip.best_iteration + 1} árboles bip, {time.time() - t0:.0f}s\")\n",
    "    np.savez(os.path.join(CARPETA_MODELOS, f\"{capa}_oof.npz\"), prob=prob_oof, bip=bip_oof)\n",
    "    return prob_oof, bip_oof, modelos\n",
    "\n",
    "def cargar_capa(capa):\n",
    "    arch = np.load(os.path.join(CARPETA_MODELOS, f\"{capa}_oof.npz\"))\n",
    "    modelos = []\n",
    "    for i in range(len(bloques)):\n",
    "        m_res, m_bip = xgb.XGBClassifier(), xgb.XGBRegressor()\n",
    "        m_res.load_model(os.path.join(CARPETA_MODELOS, f\"{capa}_resultado_{i}.json\"))\n",
    "        m_bip.load_model(os.path.join(CARPETA_MODELOS, f\"{capa}_bip_{i}.json\"))\n",
    "        modelos.append((m_res, m_bip))\n",
    "    return arch['prob'], arch['bip'], modelos"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 13,
   "id": "fdc91af1",
   "metadata": {},
   "outputs": [],
   "source": [
    "OOF, MODELOS = {}, {}\n",
    "for capa in VARIABLES:\n",
    "    ya_existe = os.path.exists(os.path.join(CARPETA_MODELOS, f\"{capa}_oof.npz\"))\n",
    "    if REENTRENAR or not ya_existe:\n",
    "        print(f\"Entrenando {capa}...\")\n",
    "        prob, bip, modelos = entrenar_capa(capa)\n",
    "    else:\n",
    "        print(f\"Cargando {capa} desde disco\")\n",
    "        prob, bip, modelos = cargar_capa(capa)\n",
    "    OOF[capa] = {'prob': prob, 'bip': bip}\n",
    "    MODELOS[capa] = modelos"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6533c7de",
   "metadata": {},
   "source": [
    "## Paso 5: Carreras esperadas (xRV) y escala Plus"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 14,
   "id": "d45c000d",
   "metadata": {},
   "outputs": [],
   "source": [
    "def matriz_rv(capa, bip_pred, balls, strikes, v_conteo):\n",
    "    n = len(bip_pred)\n",
    "    R = np.zeros((n, len(CLASES)), dtype=np.float64)\n",
    "    if capa == 'stuff':\n",
    "        for k, c in enumerate(CLASES[:-1]):\n",
    "            R[:, k] = rv_clase_liga.filter(pl.col('resultado') == c)['rv_pitcheo'][0]\n",
    "        R[:, -1] = bip_pred - v_conteo_bip_liga\n",
    "    else:\n",
    "        for k, c in enumerate(CLASES[:-1]):\n",
    "            vals = rv_clase_conteo.filter((pl.col('resultado') == c) & \n",
    "                                          (pl.col('Balls').is_in(balls)) & \n",
    "                                          (pl.col('Strikes').is_in(strikes)))['rv_pitcheo'].to_numpy()\n",
    "            R[:, k] = vals\n",
    "        R[:, -1] = bip_pred - v_conteo\n",
    "    return R\n",
    "\n",
    "def calcular_xrv(capa, prob, bip_pred):\n",
    "    R = matriz_rv(capa, bip_pred, d['Balls'].to_numpy(), d['Strikes'].to_numpy(), d['v_conteo'].to_numpy())\n",
    "    return (prob * R).sum(axis=1)\n",
    "\n",
    "ESCALA = {}\n",
    "for capa in VARIABLES:\n",
    "    d = d.with_columns(**{f'xrv_{capa}': calcular_xrv(capa, OOF[capa]['prob'], OOF[capa]['bip'])})\n",
    "    puntaje = -d[f'xrv_{capa}']\n",
    "    ESCALA[capa] = puntaje.group_by(d['year']).agg([pl.col(f'xrv_{capa}').mean().alias('mean'), \n",
    "                                                     pl.col(f'xrv_{capa}').std().alias('std')])\n",
    "    d = d.join(ESCALA[capa], on='year', how='left', suffix='_esc')\n",
    "    d = d.with_columns(**{f'{capa}_plus': 100 + 10 * (puntaje - pl.col(f'xrv_{capa}_esc_mean')) / pl.col(f'xrv_{capa}_esc_std')})\n",
    "\n",
    "for k, c in enumerate(CLASES):\n",
    "    d = d.with_columns(**{f'p_{c}': OOF['pitching']['prob'][:, k]})\n",
    "\n",
    "print(d.select(['stuff_plus', 'location_plus', 'pitching_plus']).describe())"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "8a1b2c3d",
   "metadata": {},
   "source": [
    "## Paso 8: Bayesian Resolvability (Tu aportación original)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 15,
   "id": "8a1b2c3e",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Modelo Bayesiano de Resolvabilidad Perceptual\n",
    "# Basado en Bayesball (bioRxiv 2022): el bateador integra prior + likelihood\n",
    "# En altitud, menor densidad del aire → menos efecto Magnus → trayectoria más predecible\n",
    "# → varianza del likelihood disminuye → bateador confía más en observación → pitch más \"resoluble\"\n",
    "\n",
    "# Calcular prior del bateador (movimiento esperado por tipo de pitch)\n",
    "prior = d.group_by('tipo').agg([\n",
    "    pl.col('InducedVertBreak').mean().alias('ivb_mean'),\n",
    "    pl.col('InducedVertBreak').std().alias('ivb_std'),\n",
    "    pl.col('hb').mean().alias('hb_mean'),\n",
    "    pl.col('hb').std().alias('hb_std'),\n",
    "])\n",
    "\n",
    "d = d.join(prior, on='tipo', how='left')\n",
    "\n",
    "# Calcular discrepancia (z-score) entre observación y prior\n",
    "d = d.with_columns([\n",
    "    z_ivb=(pl.col('InducedVertBreak') - pl.col('ivb_mean')) / pl.col('ivb_std'),\n",
    "    z_hb=(pl.col('hb') - pl.col('hb_mean')) / pl.col('hb_std'),\n",
    "])\n",
    "\n",
    "# Varianza del likelihood con factor de altitud (0.75 en Extreme Altitude)\n",
    "ALTITUDE_FACTOR = 0.75\n",
    "d = d.with_columns([\n",
    "    likelihood_var_ivb=pl.when(pl.col('altitud') == 2)\n",
    "                         .then(pl.col('ivb_std')**2 * ALTITUDE_FACTOR)\n",
    "                         .otherwise(pl.col('ivb_std')**2),\n",
    "    likelihood_var_hb=pl.when(pl.col('altitud') == 2)\n",
    "                        .then(pl.col('hb_std')**2 * ALTITUDE_FACTOR)\n",
    "                        .otherwise(pl.col('hb_std')**2),\n",
    "])\n",
    "\n",
    "# Varianza posterior: 1/σ_post² = 1/σ_p² + 1/σ_l²\n",
    "d = d.with_columns([\n",
    "    posterior_var_ivb=1 / (1/pl.col('ivb_std')**2 + 1/pl.col('likelihood_var_ivb')),\n",
    "    posterior_var_hb=1 / (1/pl.col('hb_std')**2 + 1/pl.col('likelihood_var_hb')),\n",
    "])\n",
    "\n",
    "# Resolvability score: fracción de incertidumbre del prior colapsada\n",
    "d = d.with_columns([\n",
    "    resolv_ivb=0.5 * (1 - pl.col('posterior_var_ivb') / pl.col('ivb_std')**2),\n",
    "    resolv_hb=0.5 * (1 - pl.col('posterior_var_hb') / pl.col('hb_std')**2),\n",
    "    resolvability_score=(0.5 * (1 - pl.col('posterior_var_ivb') / pl.col('ivb_std')**2) + \n",
    "                         0.5 * (1 - pl.col('posterior_var_hb') / pl.col('hb_std')**2)) / 2,\n",
    "])\n",
    "\n",
    "# Interacciones con altitud\n",
    "d = d.with_columns([\n",
    "    resolv_x_extreme=pl.col('resolvability_score') * (pl.col('altitud') == 2).cast(pl.Float64),\n",
    "    resolv_x_noalt=pl.col('resolvability_score') * (pl.col('altitud') == 0).cast(pl.Float64),\n",
    "    resolv_x_medium=pl.col('resolvability_score') * (pl.col('altitud') == 1).cast(pl.Float64),\n",
    "])\n",
    "\n",
    "print(\"Resolvability score por altitud:\")\n",
    "print(d.group_by('altitud').agg(pl.col('resolvability_score').mean()).sort('altitud'))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 16,
   "id": "8a1b2c3f",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Agregar features bayesianas al modelo de Stuff+\n",
    "VARIABLES['stuff'].extend(['resolvability_score', 'resolv_x_extreme', 'resolv_x_noalt', 'resolv_x_medium'])\n",
    "VARIABLES['pitching'].extend(['resolvability_score', 'resolv_x_extreme', 'resolv_x_noalt', 'resolv_x_medium'])\n",
    "\n",
    "# Reentrenar con las nuevas features si es necesario\n",
    "if REENTRENAR:\n",
    "    OOF, MODELOS = {}, {}\n",
    "    for capa in VARIABLES:\n",
    "        print(f\"Reentrenando {capa} con features bayesianas...\")\n",
    "        prob, bip, modelos = entrenar_capa(capa)\n",
    "        OOF[capa] = {'prob': prob, 'bip': bip}\n",
    "        MODELOS[capa] = modelos\n",
    "    \n",
    "    # Recalcular xRV y Plus\n",
    "    for capa in VARIABLES:\n",
    "        d = d.with_columns(**{f'xrv_{capa}': calcular_xrv(capa, OOF[capa]['prob'], OOF[capa]['bip'])})\n",
    "        puntaje = -d[f'xrv_{capa}']\n",
    "        d = d.join(ESCALA[capa], on='year', how='left', suffix='_esc')\n",
    "        d = d.with_columns(**{f'{capa}_plus': 100 + 10 * (puntaje - pl.col(f'xrv_{capa}_esc_mean')) / pl.col(f'xrv_{capa}_esc_std')})\n",
    "\n",
    "print(\"✅ Modelo Bayesian Resolvability integrado\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "fec2a75f",
   "metadata": {},
   "source": [
    "## Paso 9: Agregación y respuestas del entregable"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 17,
   "id": "27719927",
   "metadata": {},
   "outputs": [],
   "source": [
    "COLS_PLUS = ['stuff_plus', 'location_plus', 'pitching_plus']\n",
    "COLS_PROB = [f'p_{c}' for c in CLASES]\n",
    "\n",
    "por_pitcher_tipo = (\n",
    "    d.group_by(['pitcher_anon_id', 'year', 'tipo'])\n",
    "    .agg([\n",
    "        pl.col('stuff_plus').count().alias('pitcheos'),\n",
    "        pl.col('uso_pct').first(),\n",
    "        pl.col('RelSpeed').mean().alias('velo'),\n",
    "        pl.col('InducedVertBreak').mean().alias('ivb'),\n",
    "        pl.col('hb').mean().alias('hb'),\n",
    "    ] + [pl.col(c).mean().alias(c) for c in COLS_PLUS + COLS_PROB])\n",
    "    .filter(pl.col('pitcheos') >= 50)\n",
    ")\n",
    "\n",
    "print(\"Top 15 pitchers por tipo en CDMX:\")\n",
    "print(por_pitcher_tipo.sort('stuff_plus', descending=True).head(15))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 18,
   "id": "7b7adfb4",
   "metadata": {},
   "outputs": [],
   "source": [
    "por_pitcher = (\n",
    "    d.group_by(['pitcher_anon_id', 'year'])\n",
    "    .agg([\n",
    "        pl.col('stuff_plus').count().alias('pitcheos'),\n",
    "        pl.col('PitcherThrows').first().alias('mano'),\n",
    "    ] + [pl.col(c).mean().alias(c) for c in COLS_PLUS])\n",
    "    .filter(pl.col('pitcheos') >= 300)\n",
    ")\n",
    "\n",
    "print(\"Mejor Stuff+ calificado en CDMX:\")\n",
    "print(por_pitcher.sort('stuff_plus', descending=True).head(10))\n",
    "\n",
    "print(\"\\nMás afectados por la altitud (menor Stuff+ en CDMX):\")\n",
    "print(por_pitcher.sort('stuff_plus').head(10))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 19,
   "id": "8c8f4cae",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Arsenal: combinación de tipos que el pitcher usa al menos 10% del tiempo\n",
    "USO_MINIMO = 0.10\n",
    "uso = por_pitcher_tipo.filter(pl.col('uso_pct') >= USO_MINIMO)\n",
    "arsenal = uso.group_by(['pitcher_anon_id', 'year']).agg(\n",
    "    pl.col('tipo').sort().unique().str.join(' + ').alias('arsenal')\n",
    ")\n",
    "por_pitcher = por_pitcher.join(arsenal, on=['pitcher_anon_id', 'year'], how='left')\n",
    "\n",
    "por_arsenal = (\n",
    "    por_pitcher.filter(pl.col('arsenal').is_not_null())\n",
    "    .group_by('arsenal')\n",
    "    .agg([\n",
    "        pl.col('stuff_plus').count().alias('pitcher_temporadas'),\n",
    "        pl.col('stuff_plus').mean().alias('stuff_cdmx'),\n",
    "    ])\n",
    "    .filter(pl.col('pitcher_temporadas') >= 8)\n",
    ")\n",
    "\n",
    "print(\"Arsenales con mayor Stuff+ en CDMX:\")\n",
    "print(por_arsenal.sort('stuff_cdmx', descending=True).head(15))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 20,
   "id": "591fd196",
   "metadata": {},
   "outputs": [],
   "source": [
    "# Exportar resultados\n",
    "cols_pitcheo = ['PitchUID', 'year', 'game_anon_id', 'pitcher_anon_id', 'tipo', 'altitude_category',\n",
    "                'Balls', 'Strikes', 'RelSpeed', 'InducedVertBreak', 'hb', 'loc_x', 'PlateLocHeight',\n",
    "                'xrv_stuff', 'xrv_location', 'xrv_pitching', 'rv_pitcheo', 'resolvability_score'] + COLS_PLUS + COLS_PROB\n",
    "\n",
    "d.select(cols_pitcheo).write_pickle(os.path.join(CARPETA_RESULTADOS, \"stuff_plus_por_pitcheo.pkl\"))\n",
    "por_pitcher_tipo.write_csv(os.path.join(CARPETA_RESULTADOS, \"stuff_plus_por_pitcher_tipo.csv\"))\n",
    "por_pitcher.write_csv(os.path.join(CARPETA_RESULTADOS, \"stuff_plus_por_pitcher.csv\"))\n",
    "por_arsenal.write_csv(os.path.join(CARPETA_RESULTADOS, \"stuff_plus_por_arsenal.csv\"))\n",
    "\n",
    "print(\"✅ Archivos exportados en resultados/\")\n",
    "print(os.listdir(CARPETA_RESULTADOS))"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.12.1"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

{'cells': [{'cell_type': 'markdown',
   'id': '983f910a',
   'metadata': {},
   'source': ['# Stuff+ / Location+ / Pitching+ con Bayesian Resolvability — Diablos Rojos (ISAC 2026)\n',
    '\n',
    '**Entregable 1:** Stuff+ por pitcheo, pitcher y arsenal, y comparación CDMX (altitud extrema) vs. otros estadios.\n',
    '\n',
    '**Innovación:** Integración del modelo Bayesball de resolvabilidad perceptual del bateador en altitud.\n',
    '\n',
    '| Paso | Qué hace |\n',
    '|---|---|\n',
    '| 0 | Configuración y carga de datos con Polars |\n',
    '| 1 | Valor en carreras de cada pitcheo (según el conteo) |\n',
    '| 2 | Variables: físicas, de arsenal, de contexto y bayesianas |\n',
    '| 3 | Variable objetivo descompuesta: 5 resultados posibles + calidad de bola en juego |\n',
    '| 4 | Entrenamiento XGBoost con validación *pitcher holdout* |\n',
    '| 5 | Carreras esperadas (xRV) → escala 100 + 10·z |\n',
    '| 6 | Validación: calidad, estabilidad, capacidad predictiva |\n